# 🚀 Servidor OpenCode en Google Colab con CCUs (Google AI Pro)

Este cuaderno te permite ejecutar modelos de IA de código abierto de máxima potencia (**Qwen 2.5 Coder** o **DeepSeek Coder**) utilizando las **Compute Units (CCUs)** de tu suscripción de **Google AI Pro** sobre GPUs dedicadas en la nube de Google (T4, L4 o A100).

---

### ⚡ Instrucciones de inicio:
1. En el menú superior de Google Colab, andá a **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y asegurate de tener seleccionada **GPU** (ej: T4, L4 o A100).
2. Ejecutá las celdas en orden (o presioná `Ctrl + F9` para ejecutar todo).
3. En la última celda aparecerá tu **URL pública de Cloudflare** (ej: `https://xxxx-xxxx.trycloudflare.com`).
4. Copiá esa URL y pegala en el chat de Antigravity o decile: *"Conectate a https://..."* para que el servidor MCP empiece a usar este modelo al instante.

In [ ]:
# 1. Verificar GPU de Google Colab asignada
!nvidia-smi

In [ ]:
# 2. Instalar Ollama y Cloudflare Tunnel
import os, sys, time, subprocess

print("📦 Instalando Ollama en la máquina virtual de Google...")
!curl -fsSL https://ollama.ai/install.sh | sh

print("🌐 Descargando Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("✅ Instalación completada con éxito.")

In [ ]:
# 3. Seleccionar Modelo e Iniciar Demonio de Ollama
# Opciones recomendadas:
# • "qwen2.5-coder:7b"  -> Ultrarrápido, excelente para TypeScript, React, APIs y lógica.
# • "qwen2.5-coder:14b" -> Mayor razonamiento (usar con GPU L4 o A100 si tenés CCUs altas).
# • "deepseek-coder:6.7b" -> Modelo clásico de DeepSeek.
# • "deepseek-coder-v2:16b-lite" -> DeepSeek Coder V2 versión Lite MoE.

MODELO = "qwen2.5-coder:7b" #@param ["qwen2.5-coder:7b", "qwen2.5-coder:14b", "deepseek-coder:6.7b", "deepseek-coder-v2:16b-lite"]

print("⚡ Reiniciando procesos previos de Ollama...")
!pkill -f "ollama serve" || true
time.sleep(2)

# Iniciar servidor Ollama escuchando en todas las interfaces
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"])
time.sleep(4)

print(f"📥 Descargando modelo '{MODELO}' en la GPU...")
!ollama pull {MODELO}

print(f"\n🎉 ¡Modelo {MODELO} listo y cargado en VRAM!")

In [ ]:
# 4. Iniciar Túnel Seguro y Exponer Endpoint Público para el Servidor MCP
import re, time, subprocess

print("🚀 Levantando Cloudflare Tunnel seguro...")
!pkill -f "cloudflared" || true

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()

while time.time() - start_time < 35:
    line = tunnel.stdout.readline()
    if not line: continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*65)
    print("🟢 ¡TU SERVIDOR OPENCODE ESTÁ EN VIVO EN LA NUBE DE GOOGLE!")
    print("="*65)
    print(f"\n👉 URL DEL TÚNEL PARA COPIAR:")
    print(f"   {public_url}\n")
    print(f"👉 MODELO ACTIVO: {MODELO}")
    print("="*65)
    print("\n📋 CÓMO VINCULARLO CON ANTIGRAVITY:")
    print("Simplemente escribí en el chat:")
    print(f'   "Conectate al endpoint {public_url}"')
    print("="*65 + "\n")
    
    # Mantener activa la celda sin bloquear
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("Túnel finalizado por el usuario.")
else:
    print("⚠️ No se pudo obtener la URL del túnel. Revisá los registros de cloudflared:")
    print(tunnel.communicate()[0])